In [ ]:
# ============================================================
# Setup: Import all required libraries
# ============================================================

import numpy as np
import pandas as pd
from scipy import optimize, stats
from scipy.integrate import odeint
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import networkx as nx
from tqdm import tqdm
from itertools import combinations
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12

# Reproducibility
np.random.seed(42)

print("All libraries loaded successfully.")
print(f"NumPy: {np.__version__}, Pandas: {pd.__version__}")

---

# Part 1: Game Theory Foundations

Game theory provides the mathematical framework for analyzing strategic interactions between rational agents. In blockchain systems, every participant -- miners, validators, users -- is making strategic decisions that affect everyone else.

## The Prisoner's Dilemma

The Prisoner's Dilemma is the canonical game theory problem. Two players simultaneously choose to **Cooperate** (C) or **Defect** (D). The payoff matrix is:

|  | Player B: C | Player B: D |
|---|---|---|
| **Player A: C** | (3, 3) | (0, 5) |
| **Player A: D** | (5, 0) | (1, 1) |

The dilemma: mutual cooperation yields the best *collective* outcome, but each individual has an incentive to defect. In blockchain, this maps directly to mining honesty: following the protocol (cooperating) benefits the network, but an individual miner might profit more by deviating.

In [ ]:
# ============================================================
# 1.1 Prisoner's Dilemma Payoff Matrix
# ============================================================

# Payoff matrix: payoffs[i][j] = (payoff_A, payoff_B)
# Actions: 0 = Cooperate, 1 = Defect
COOPERATE, DEFECT = 0, 1

# Standard PD payoffs: T > R > P > S and 2R > T + S
R, S, T, P = 3, 0, 5, 1  # Reward, Sucker, Temptation, Punishment

payoff_matrix_A = np.array([[R, S],
                             [T, P]])

payoff_matrix_B = np.array([[R, T],
                             [S, P]])

print("Prisoner's Dilemma Payoff Matrix")
print("================================")
print(f"                Player B: C    Player B: D")
print(f"Player A: C     ({R}, {R})          ({S}, {T})")
print(f"Player A: D     ({T}, {S})          ({P}, {P})")
print()
print(f"Conditions check:")
print(f"  T > R > P > S: {T} > {R} > {P} > {S} = {T > R > P > S}")
print(f"  2R > T + S:    {2*R} > {T+S} = {2*R > T + S}")

In [ ]:
# ============================================================
# 1.2 Strategies for Iterated Prisoner's Dilemma
# ============================================================

def always_cooperate(history_self, history_opponent):
    """Always cooperate regardless of opponent's actions."""
    return COOPERATE

def always_defect(history_self, history_opponent):
    """Always defect regardless of opponent's actions."""
    return DEFECT

def tit_for_tat(history_self, history_opponent):
    """Cooperate first, then copy opponent's last move."""
    if len(history_opponent) == 0:
        return COOPERATE
    return history_opponent[-1]

def random_strategy(history_self, history_opponent):
    """Cooperate or defect with equal probability."""
    return np.random.choice([COOPERATE, DEFECT])

strategies = {
    'Always Cooperate': always_cooperate,
    'Always Defect': always_defect,
    'Tit-for-Tat': tit_for_tat,
    'Random': random_strategy
}

print("Strategies defined:")
for name in strategies:
    print(f"  - {name}")

In [ ]:
# ============================================================
# 1.3 Iterated PD Simulation Engine
# ============================================================

def play_iterated_pd(strategy_a, strategy_b, rounds=200):
    """Play an iterated Prisoner's Dilemma between two strategies.
    
    Returns:
        scores_a: cumulative scores for player A at each round
        scores_b: cumulative scores for player B at each round
    """
    history_a, history_b = [], []
    scores_a, scores_b = [], []
    cum_a, cum_b = 0, 0
    
    for _ in range(rounds):
        action_a = strategy_a(history_a, history_b)
        action_b = strategy_b(history_b, history_a)
        
        cum_a += payoff_matrix_A[action_a, action_b]
        cum_b += payoff_matrix_B[action_a, action_b]
        
        history_a.append(action_a)
        history_b.append(action_b)
        scores_a.append(cum_a)
        scores_b.append(cum_b)
    
    return np.array(scores_a), np.array(scores_b)


def run_tournament(strategies, rounds=200):
    """Run a round-robin tournament between all strategies."""
    names = list(strategies.keys())
    n = len(names)
    total_scores = {name: 0 for name in names}
    match_results = {}
    
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            scores_a, scores_b = play_iterated_pd(
                strategies[names[i]], strategies[names[j]], rounds
            )
            total_scores[names[i]] += scores_a[-1]
            match_results[(names[i], names[j])] = (scores_a, scores_b)
    
    return total_scores, match_results


# Run the tournament
np.random.seed(42)
total_scores, match_results = run_tournament(strategies, rounds=200)

print("Tournament Results (200 rounds, round-robin)")
print("============================================")
for name, score in sorted(total_scores.items(), key=lambda x: -x[1]):
    print(f"  {name:20s}: {score:6.0f} points")

In [ ]:
# ============================================================
# 1.4 Plot Cumulative Scores: Tit-for-Tat vs All Others
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
opponents = ['Always Cooperate', 'Always Defect', 'Random']
colors_tft = '#2196F3'
colors_opp = ['#4CAF50', '#F44336', '#FF9800']

for idx, opp in enumerate(opponents):
    ax = axes[idx]
    key = ('Tit-for-Tat', opp)
    scores_tft, scores_opp = match_results[key]
    rounds = np.arange(1, len(scores_tft) + 1)
    
    ax.plot(rounds, scores_tft, label='Tit-for-Tat', color=colors_tft, linewidth=2)
    ax.plot(rounds, scores_opp, label=opp, color=colors_opp[idx], linewidth=2, linestyle='--')
    ax.set_title(f'TfT vs {opp}', fontsize=12)
    ax.set_xlabel('Round')
    ax.set_ylabel('Cumulative Score')
    ax.legend(fontsize=9)

plt.suptitle('Iterated Prisoner\'s Dilemma: Tit-for-Tat Matchups', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Tournament bar chart
fig, ax = plt.subplots(figsize=(8, 5))
sorted_scores = sorted(total_scores.items(), key=lambda x: -x[1])
names_sorted = [s[0] for s in sorted_scores]
vals_sorted = [s[1] for s in sorted_scores]
bar_colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

ax.barh(names_sorted[::-1], vals_sorted[::-1], color=bar_colors[::-1])
ax.set_xlabel('Total Tournament Score')
ax.set_title('Prisoner\'s Dilemma Tournament Results (200 rounds)')
for i, v in enumerate(vals_sorted[::-1]):
    ax.text(v + 20, i, f'{v:.0f}', va='center', fontsize=11)
plt.tight_layout()
plt.show()

### Blockchain Connection: Mining as a Cooperation Game

In Proof-of-Work mining, the honest strategy is analogous to **cooperation**: following the protocol, building on the longest chain, and broadcasting blocks immediately. **Defection** corresponds to selfish mining -- withholding blocks, attempting double spends, or performing 51% attacks.

The key insight from Axelrod's tournaments (and our simulation above) is that **Tit-for-Tat** -- a simple, retaliatory, forgiving strategy -- tends to perform best in iterated games. Blockchain protocols encode a form of Tit-for-Tat through:

- **Longest-chain rule**: honest miners naturally "retaliate" against attackers by orphaning their blocks
- **Difficulty adjustment**: the protocol adapts to maintain cooperation incentives
- **Slashing (PoS)**: direct punishment for detected misbehavior

---

# Part 2: Nash Equilibrium

A **Nash Equilibrium** is a strategy profile where no player can improve their payoff by unilaterally changing their strategy. Finding Nash Equilibria tells us what rational agents *will* do -- which is essential for protocol design.

## Nash Equilibrium Finder for 2x2 Games

For a 2x2 game, we can find:
1. **Pure strategy NE**: where each player plays a single action
2. **Mixed strategy NE**: where players randomize with specific probabilities

In [ ]:
# ============================================================
# 2.1 Nash Equilibrium Finder for 2x2 Games
# ============================================================

def find_nash_equilibria_2x2(payoff_A, payoff_B):
    """
    Find all Nash Equilibria (pure and mixed) for a 2x2 game.
    
    payoff_A[i,j] = payoff to player A when A plays i, B plays j
    payoff_B[i,j] = payoff to player B when A plays i, B plays j
    """
    equilibria = []
    
    # --- Pure Strategy NE ---
    for i in range(2):
        for j in range(2):
            # Check if i is best response for A given B plays j
            a_br = payoff_A[i, j] >= payoff_A[1-i, j]
            # Check if j is best response for B given A plays i
            b_br = payoff_B[i, j] >= payoff_B[i, 1-j]
            if a_br and b_br:
                actions = ['C', 'D']
                equilibria.append({
                    'type': 'Pure',
                    'strategy_A': actions[i],
                    'strategy_B': actions[j],
                    'payoff_A': payoff_A[i, j],
                    'payoff_B': payoff_B[i, j]
                })
    
    # --- Mixed Strategy NE ---
    # Player A mixes to make B indifferent: q*B[0,0] + (1-q)*B[0,1] = q*B[1,0] + (1-q)*B[1,1]
    # Player B mixes to make A indifferent: p*A[0,0] + (1-p)*A[0,1] = p*A[1,0] + (1-p)*A[1,1]
    denom_p = (payoff_A[0,0] - payoff_A[0,1] - payoff_A[1,0] + payoff_A[1,1])
    denom_q = (payoff_B[0,0] - payoff_B[1,0] - payoff_B[0,1] + payoff_B[1,1])
    
    if denom_p != 0 and denom_q != 0:
        p = (payoff_A[1,1] - payoff_A[0,1]) / denom_p  # prob A plays action 0
        q = (payoff_B[1,1] - payoff_B[0,1]) / denom_q  # prob B plays action 0
        
        if 0 < p < 1 and 0 < q < 1:
            exp_A = p * q * payoff_A[0,0] + p*(1-q)*payoff_A[0,1] + \
                    (1-p)*q*payoff_A[1,0] + (1-p)*(1-q)*payoff_A[1,1]
            exp_B = p * q * payoff_B[0,0] + p*(1-q)*payoff_B[0,1] + \
                    (1-p)*q*payoff_B[1,0] + (1-p)*(1-q)*payoff_B[1,1]
            equilibria.append({
                'type': 'Mixed',
                'strategy_A': f'C:{p:.3f}, D:{1-p:.3f}',
                'strategy_B': f'C:{q:.3f}, D:{1-q:.3f}',
                'payoff_A': exp_A,
                'payoff_B': exp_B
            })
    
    return equilibria


# Test with standard Prisoner's Dilemma
pd_eq = find_nash_equilibria_2x2(payoff_matrix_A, payoff_matrix_B)
print("Nash Equilibria for Standard Prisoner's Dilemma:")
print("=" * 50)
for eq in pd_eq:
    print(f"  Type: {eq['type']}")
    print(f"  Player A: {eq['strategy_A']}, Player B: {eq['strategy_B']}")
    print(f"  Payoffs: A={eq['payoff_A']:.2f}, B={eq['payoff_B']:.2f}")
    print()

In [ ]:
# ============================================================
# 2.2 Honest vs Selfish Mining Payoff Matrix
# ============================================================

def mining_payoff_matrix(alpha, gamma=0.5):
    """
    Construct payoff matrix for honest vs selfish mining.
    
    alpha: attacker's fraction of total hash power
    gamma: fraction of honest miners that mine on attacker's block during a race
    
    Actions: 0 = Honest, 1 = Selfish
    Payoffs represent relative revenue per unit hash power.
    """
    # Honest mining revenue = proportional to hash power
    honest_revenue = alpha  # normalized
    
    # Selfish mining revenue (from Eyal & Sirer 2014 approximation)
    # Revenue when selfish mining against honest majority
    numerator = alpha * (1 - alpha)**2 * (4*alpha + gamma*(1 - 2*alpha)) - alpha**3
    denominator = 1 - alpha * (1 + (2 - alpha) * alpha)
    
    if denominator != 0 and alpha > 0:
        selfish_revenue = alpha + numerator / denominator
    else:
        selfish_revenue = alpha
    
    # Payoff matrix (attacker perspective)
    # [honest_vs_honest, honest_vs_selfish_other]
    # [selfish_vs_honest, selfish_vs_selfish]
    A = np.array([
        [alpha, alpha * 0.9],           # Honest: proportional, slightly less if other is selfish
        [selfish_revenue, alpha * 0.7]  # Selfish: boosted vs honest, worse if both selfish
    ])
    
    B = A.T  # Symmetric game
    
    return A, B, selfish_revenue


# Analyze for different hash power levels
print("Selfish Mining Analysis")
print("=" * 60)
print(f"{'Hash Power':>12s}  {'Honest Rev':>12s}  {'Selfish Rev':>12s}  {'Rational?':>10s}")
print("-" * 60)

alphas = np.arange(0.05, 0.55, 0.05)
honest_revs = []
selfish_revs = []

for alpha in alphas:
    A, B, selfish_rev = mining_payoff_matrix(alpha)
    honest_rev = alpha
    is_rational = selfish_rev > honest_rev
    honest_revs.append(honest_rev)
    selfish_revs.append(selfish_rev)
    marker = "  <-- YES" if is_rational else ""
    print(f"{alpha:>11.0%}  {honest_rev:>12.4f}  {selfish_rev:>12.4f}  {marker}")

In [ ]:
# ============================================================
# 2.3 Selfish Mining Threshold Visualization
# ============================================================

alphas_fine = np.linspace(0.01, 0.50, 200)
honest_fine = alphas_fine.copy()
selfish_fine = []

for alpha in alphas_fine:
    _, _, sr = mining_payoff_matrix(alpha, gamma=0.5)
    selfish_fine.append(sr)

selfish_fine = np.array(selfish_fine)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(alphas_fine * 100, honest_fine, label='Honest Mining Revenue', 
        color='#4CAF50', linewidth=2.5)
ax.plot(alphas_fine * 100, selfish_fine, label='Selfish Mining Revenue', 
        color='#F44336', linewidth=2.5)

# Find crossover point
crossover_idx = np.argmin(np.abs(selfish_fine - honest_fine))
crossover_alpha = alphas_fine[crossover_idx]

ax.axvline(x=crossover_alpha * 100, color='gray', linestyle='--', alpha=0.7)
ax.fill_betweenx([0, 0.6], crossover_alpha * 100, 50, alpha=0.1, color='red',
                  label=f'Selfish mining rational (>{crossover_alpha:.0%} hash power)')

ax.set_xlabel('Attacker Hash Power (%)', fontsize=13)
ax.set_ylabel('Relative Revenue', fontsize=13)
ax.set_title('Honest vs Selfish Mining: When Does Defection Pay?', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0, 50)
ax.set_ylim(0, 0.55)
plt.tight_layout()
plt.show()

print(f"\nSelfish mining becomes rational at approximately {crossover_alpha:.1%} hash power.")
print("This is consistent with Eyal & Sirer (2014) finding of ~25% threshold.")

### Key Takeaway

The Nash Equilibrium analysis reveals that honest mining is **not** always the dominant strategy. When a miner controls more than roughly 25% of hash power, selfish mining can yield higher revenue than honest mining. This is a critical result for protocol security -- it means that the "51% attack threshold" is actually too optimistic. Protocols must be designed to make defection unprofitable at *all* levels of hash power concentration.

---

# Part 3: Staking Economics

Proof-of-Stake systems replace computational work with economic stake. Validators lock up capital and earn rewards for honest behavior -- or face **slashing** (loss of stake) for misbehavior. Understanding the economics of staking is essential for both protocol designers and participants.

## Validator Economics Model

Key parameters:
- **stake**: amount of tokens staked by this validator
- **total_stake**: total tokens staked across all validators
- **block_reward**: reward per block for the selected validator
- **slash_prob**: probability of being slashed per epoch (if behaving dishonestly)
- **slash_penalty**: fraction of stake lost when slashed

In [ ]:
# ============================================================
# 3.1 Validator Economics: Honest vs Dishonest Returns
# ============================================================

def validator_expected_return(stake, total_stake, block_reward, 
                                epochs_per_year=365,
                                slash_prob=0.0, slash_penalty=0.0,
                                dishonest_bonus=0.0):
    """
    Calculate expected annual return for a validator.
    
    Parameters:
        stake: validator's stake
        total_stake: total network stake
        block_reward: reward per block/epoch
        epochs_per_year: number of epochs in a year
        slash_prob: probability of slashing per epoch (0 for honest)
        slash_penalty: fraction of stake lost if slashed
        dishonest_bonus: extra reward per epoch from dishonest behavior
    
    Returns:
        expected_annual_return, annual_yield_pct
    """
    selection_prob = stake / total_stake
    
    # Expected reward per epoch
    reward_per_epoch = selection_prob * block_reward + dishonest_bonus
    
    # Expected slashing loss per epoch
    slash_loss_per_epoch = slash_prob * slash_penalty * stake
    
    # Net expected per epoch
    net_per_epoch = reward_per_epoch - slash_loss_per_epoch
    
    # Annual
    annual_return = net_per_epoch * epochs_per_year
    annual_yield = (annual_return / stake) * 100
    
    return annual_return, annual_yield


# Parameters
stake = 32  # ETH
total_stake = 28_000_000  # ~28M ETH staked
block_reward = 0.05  # ETH per block

# Honest validator
honest_return, honest_yield = validator_expected_return(
    stake, total_stake, block_reward
)

# Dishonest validator (small MEV extraction bonus, but risk of slashing)
dishonest_return, dishonest_yield = validator_expected_return(
    stake, total_stake, block_reward,
    slash_prob=0.01,        # 1% chance of getting caught per epoch
    slash_penalty=0.10,     # lose 10% of stake
    dishonest_bonus=0.0001  # tiny extra MEV
)

print("Validator Expected Returns (Annual)")
print("=" * 50)
print(f"Stake: {stake} ETH | Total Network Stake: {total_stake:,.0f} ETH")
print(f"Block Reward: {block_reward} ETH")
print()
print(f"{'Strategy':<15} {'Annual Return (ETH)':>20} {'Yield':>10}")
print("-" * 50)
print(f"{'Honest':<15} {honest_return:>20.4f} {honest_yield:>9.2f}%")
print(f"{'Dishonest':<15} {dishonest_return:>20.4f} {dishonest_yield:>9.2f}%")
print()
print(f"Conclusion: Honest staking yields {honest_yield - dishonest_yield:+.2f}% more annually.")
print("Slashing makes dishonest behavior economically irrational.")

In [ ]:
# ============================================================
# 3.2 Staking Yield Calculator: Nominal vs Real Yield
# ============================================================

def staking_yield(total_staked_pct, base_issuance_rate, inflation_rate=0.0):
    """
    Calculate nominal and real staking yields.
    
    Parameters:
        total_staked_pct: fraction of total supply staked (0 to 1)
        base_issuance_rate: annual token issuance as fraction of total supply
        inflation_rate: annual inflation (dilution) rate
    
    Returns:
        nominal_yield, real_yield (both as percentages)
    """
    if total_staked_pct <= 0:
        return 0.0, 0.0
    
    nominal_yield = (base_issuance_rate / total_staked_pct) * 100
    real_yield = nominal_yield - (inflation_rate * 100)
    
    return nominal_yield, real_yield


# Sweep across staking participation rates
participation_rates = np.linspace(0.05, 0.90, 100)
nominal_yields = []
real_yields = []

for pr in participation_rates:
    ny, ry = staking_yield(pr, base_issuance_rate=0.04, inflation_rate=0.04)
    nominal_yields.append(ny)
    real_yields.append(ry)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(participation_rates * 100, nominal_yields, label='Nominal Yield', 
        color='#2196F3', linewidth=2.5)
ax.plot(participation_rates * 100, real_yields, label='Real Yield (after inflation)', 
        color='#FF9800', linewidth=2.5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(participation_rates * 100, real_yields, 0, 
                where=[ry > 0 for ry in real_yields], alpha=0.1, color='green')
ax.fill_between(participation_rates * 100, real_yields, 0, 
                where=[ry <= 0 for ry in real_yields], alpha=0.1, color='red')

ax.set_xlabel('Staking Participation Rate (%)', fontsize=13)
ax.set_ylabel('Annual Yield (%)', fontsize=13)
ax.set_title('Staking Yields vs Participation Rate\n(4% base issuance, 4% inflation)', fontsize=14)
ax.legend(fontsize=12)
ax.set_xlim(5, 90)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3.3 Cross-Chain Staking Comparison: ETH, SOL, ATOM
# ============================================================

# Realistic parameters (approximate as of 2024-2025)
chains = {
    'Ethereum (ETH)': {
        'min_stake': 32,
        'total_supply': 120_000_000,
        'staked_pct': 0.27,
        'base_issuance': 0.006,   # ~0.6% annual issuance
        'inflation': 0.004,       # ~0.4% net (EIP-1559 burn offsets)
        'slash_penalty_max': 0.50,
        'hardware_cost_usd_yr': 1200,
        'token_price_usd': 3200
    },
    'Solana (SOL)': {
        'min_stake': 1,
        'total_supply': 580_000_000,
        'staked_pct': 0.65,
        'base_issuance': 0.055,   # ~5.5% initial, declining
        'inflation': 0.055,
        'slash_penalty_max': 1.00,
        'hardware_cost_usd_yr': 3600,  # Higher hardware requirements
        'token_price_usd': 140
    },
    'Cosmos (ATOM)': {
        'min_stake': 1,
        'total_supply': 390_000_000,
        'staked_pct': 0.62,
        'base_issuance': 0.10,    # 7-20% dynamic
        'inflation': 0.10,
        'slash_penalty_max': 0.05,
        'hardware_cost_usd_yr': 600,
        'token_price_usd': 9
    }
}

comparison_data = []
for name, params in chains.items():
    nom_yield, real_yield = staking_yield(
        params['staked_pct'], params['base_issuance'], params['inflation']
    )
    
    # Revenue for a single validator with min stake
    annual_revenue_tokens = params['min_stake'] * (nom_yield / 100)
    annual_revenue_usd = annual_revenue_tokens * params['token_price_usd']
    annual_profit_usd = annual_revenue_usd - params['hardware_cost_usd_yr']
    
    comparison_data.append({
        'Chain': name,
        'Min Stake': f"{params['min_stake']:,.0f}",
        'Staked %': f"{params['staked_pct']:.0%}",
        'Nominal Yield': f"{nom_yield:.1f}%",
        'Real Yield': f"{real_yield:.1f}%",
        'Max Slash': f"{params['slash_penalty_max']:.0%}",
        'HW Cost/yr': f"${params['hardware_cost_usd_yr']:,.0f}",
        'Min Stake Rev/yr (USD)': f"${annual_revenue_usd:,.0f}",
        'Profit/yr (USD)': f"${annual_profit_usd:,.0f}"
    })

df_comparison = pd.DataFrame(comparison_data)
print("Cross-Chain Staking Comparison")
print("=" * 90)
print(df_comparison.to_string(index=False))

In [ ]:
# ============================================================
# 3.4 Break-Even Analysis for Running a Validator
# ============================================================

def break_even_stake(token_price, hardware_cost_yr, nominal_yield_pct):
    """Calculate minimum stake (in tokens) to break even on validator costs."""
    if nominal_yield_pct <= 0 or token_price <= 0:
        return float('inf')
    annual_return_per_token = token_price * (nominal_yield_pct / 100)
    return hardware_cost_yr / annual_return_per_token


# Break-even analysis across token prices
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, params) in enumerate(chains.items()):
    ax = axes[idx]
    nom_yield, _ = staking_yield(params['staked_pct'], params['base_issuance'], params['inflation'])
    
    # Price range: 0.2x to 5x current price
    prices = np.linspace(params['token_price_usd'] * 0.2, 
                         params['token_price_usd'] * 5, 100)
    be_stakes = [break_even_stake(p, params['hardware_cost_usd_yr'], nom_yield) 
                 for p in prices]
    
    ax.plot(prices, be_stakes, color='#673AB7', linewidth=2.5)
    ax.axhline(y=params['min_stake'], color='red', linestyle='--', 
               label=f"Min stake: {params['min_stake']:,.0f}", alpha=0.7)
    ax.axvline(x=params['token_price_usd'], color='green', linestyle='--',
               label=f"Current: ${params['token_price_usd']:,.0f}", alpha=0.7)
    
    ax.set_xlabel('Token Price (USD)')
    ax.set_ylabel('Break-Even Stake (tokens)')
    ax.set_title(name, fontsize=12)
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=0)

plt.suptitle('Validator Break-Even Stake vs Token Price', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Staking Economics Insights

1. **Higher participation reduces yields**: As more tokens are staked, each validator's share of rewards decreases. This creates a natural equilibrium -- staking participation stabilizes where the yield equals participants' opportunity cost.

2. **Real yield can be negative**: If inflation exceeds nominal yield (which happens at high participation rates), non-stakers are diluted, but stakers may also lose purchasing power.

3. **Break-even depends heavily on scale**: Solo validators need significant stake to cover hardware costs. This economic reality drives delegation and pooling.

---

# Part 4: Network Effects

Network effects are the economic engine behind blockchain value accrual. A blockchain network becomes more valuable as more participants join -- for users (more counterparties), for validators (more fees), and for developers (larger market).

## Metcalfe's Law

**Metcalfe's Law** states that the value of a network is proportional to the square of its users:

$$V \propto n^2$$

This has been empirically validated for Bitcoin and Ethereum market capitalizations.

In [ ]:
# ============================================================
# 4.1 Metcalfe's Law: Network Value vs Users
# ============================================================

def metcalfe_value(n, k=1.0):
    """Network value under Metcalfe's Law: V = k * n^2"""
    return k * n ** 2

def odlyzko_value(n, k=1.0):
    """More conservative: V = k * n * log(n) (Odlyzko & Tilly)"""
    return k * n * np.log(np.maximum(n, 1))

def linear_value(n, k=1.0):
    """Linear value (no network effects): V = k * n"""
    return k * n

users = np.linspace(1, 1000, 500)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(users, metcalfe_value(users), label="Metcalfe: $V \\propto n^2$", 
        color='#F44336', linewidth=2.5)
ax.plot(users, odlyzko_value(users, k=100), label="Odlyzko: $V \\propto n \\log(n)$", 
        color='#FF9800', linewidth=2.5)
ax.plot(users, linear_value(users, k=500), label="Linear: $V \\propto n$", 
        color='#9E9E9E', linewidth=2.5, linestyle='--')

ax.set_xlabel('Number of Users (n)', fontsize=13)
ax.set_ylabel('Network Value (V)', fontsize=13)
ax.set_title('Network Value Scaling Laws', fontsize=14)
ax.legend(fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K' if x >= 1e3 else f'{x:.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4.2 Logistic S-Curve Network Growth
# ============================================================

def logistic_growth(t, L, k, t0):
    """
    Logistic growth model.
    L: carrying capacity (max users)
    k: growth rate
    t0: midpoint (inflection point)
    """
    return L / (1 + np.exp(-k * (t - t0)))


# Simulate network adoption over 10 years
months = np.arange(0, 120)  # 10 years in months

# Three scenarios
scenarios = {
    'Aggressive Growth': {'L': 500_000_000, 'k': 0.08, 't0': 48},
    'Moderate Growth':   {'L': 100_000_000, 'k': 0.06, 't0': 60},
    'Slow Growth':       {'L': 20_000_000,  'k': 0.04, 't0': 72}
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#F44336', '#2196F3', '#4CAF50']

for idx, (name, params) in enumerate(scenarios.items()):
    users_t = logistic_growth(months, **params)
    value_t = metcalfe_value(users_t, k=1e-12)  # Scale factor for display
    
    ax1.plot(months / 12, users_t / 1e6, label=name, 
             color=colors[idx], linewidth=2.5)
    ax2.plot(months / 12, value_t, label=name,
             color=colors[idx], linewidth=2.5)

ax1.set_xlabel('Years', fontsize=13)
ax1.set_ylabel('Users (millions)', fontsize=13)
ax1.set_title('Network Adoption (Logistic S-Curve)', fontsize=14)
ax1.legend(fontsize=11)

ax2.set_xlabel('Years', fontsize=13)
ax2.set_ylabel('Network Value (Metcalfe)', fontsize=13)
ax2.set_title('Network Value Over Time', fontsize=14)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4.3 Two-Sided Market Simulation (Validators + Users)
# ============================================================

def simulate_two_sided_market(n_periods=100, 
                                init_validators=10, 
                                init_users=100,
                                user_growth_sensitivity=0.5,
                                validator_growth_sensitivity=0.3,
                                max_validators=500,
                                max_users=100_000):
    """
    Simulate a two-sided blockchain market.
    
    Users attract validators (more fees to earn).
    Validators attract users (better security/throughput).
    """
    validators = np.zeros(n_periods)
    users = np.zeros(n_periods)
    validators[0] = init_validators
    users[0] = init_users
    
    for t in range(1, n_periods):
        # Users grow based on validator count (security/capacity signal)
        user_attraction = user_growth_sensitivity * np.log1p(validators[t-1])
        users[t] = min(
            users[t-1] * (1 + 0.02 + user_attraction * 0.01) + np.random.normal(0, 10),
            max_users
        )
        users[t] = max(users[t], 1)
        
        # Validators grow based on user count (fee revenue signal)
        val_attraction = validator_growth_sensitivity * np.log1p(users[t-1] / 100)
        validators[t] = min(
            validators[t-1] * (1 + 0.01 + val_attraction * 0.005) + np.random.normal(0, 0.5),
            max_validators
        )
        validators[t] = max(validators[t], 1)
    
    return validators, users


np.random.seed(42)
validators, users = simulate_two_sided_market(n_periods=150)

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

line1, = ax1.plot(range(150), users, color='#2196F3', linewidth=2.5, label='Users')
line2, = ax2.plot(range(150), validators, color='#FF9800', linewidth=2.5, label='Validators')

ax1.set_xlabel('Time Period', fontsize=13)
ax1.set_ylabel('Users', color='#2196F3', fontsize=13)
ax2.set_ylabel('Validators', color='#FF9800', fontsize=13)
ax1.set_title('Two-Sided Market Dynamics: Users and Validators', fontsize=14)
ax1.legend(handles=[line1, line2], loc='upper left', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Final state: {users[-1]:,.0f} users, {validators[-1]:.0f} validators")
print(f"Growth factor: users {users[-1]/users[0]:.1f}x, validators {validators[-1]/validators[0]:.1f}x")

### Network Effects Insights

1. **Metcalfe's Law** creates powerful winner-take-most dynamics. A network with 10x the users has ~100x the value, creating enormous barriers to entry for competitors.

2. **S-curve adoption** means growth appears slow initially, accelerates rapidly through the inflection point, then saturates. Timing market entry is critical.

3. **Two-sided markets** create a "chicken and egg" bootstrapping problem: you need validators for security, but validators need users for revenue. Most successful chains solve this with initial subsidies (high block rewards).

---

# Part 5: Agent-Based Proof-of-Stake Simulation

Agent-based models (ABMs) simulate the emergent behavior of systems by modeling individual agents and their interactions. Here we build a simple ABM of a PoS network where validators have different strategies (honest, occasionally malicious, or aggressively malicious) and experience the consequences.

## Model Setup

- **50 validators** with varying initial stakes
- **3 strategies**: Honest, Occasionally Malicious (5% defect rate), Aggressively Malicious (20% defect rate)
- **Proportional selection**: probability of being chosen as block producer is proportional to stake
- **Rewards**: honest block production earns a reward
- **Slashing**: malicious behavior detected with some probability; if caught, a fraction of stake is slashed
- **Ejection**: validators with stake below a threshold are ejected

In [ ]:
# ============================================================
# 5.1 Agent-Based PoS Simulation: Validator Class
# ============================================================

class Validator:
    def __init__(self, vid, stake, strategy='honest'):
        self.vid = vid
        self.stake = stake
        self.strategy = strategy  # 'honest', 'occasionally_malicious', 'aggressively_malicious'
        self.active = True
        self.total_rewards = 0.0
        self.total_slashed = 0.0
        self.blocks_produced = 0
        self.times_slashed = 0
    
    def decide_action(self):
        """Decide whether to act honestly or maliciously this round."""
        if self.strategy == 'honest':
            return 'honest'
        elif self.strategy == 'occasionally_malicious':
            return 'malicious' if np.random.random() < 0.05 else 'honest'
        elif self.strategy == 'aggressively_malicious':
            return 'malicious' if np.random.random() < 0.20 else 'honest'
        return 'honest'
    
    def __repr__(self):
        status = 'ACTIVE' if self.active else 'EJECTED'
        return f"V{self.vid}({self.strategy[:3]}, stake={self.stake:.1f}, {status})"


def create_validators(n_honest=30, n_occasional=12, n_aggressive=8,
                       base_stake=32.0, stake_variance=10.0):
    """Create a population of validators with different strategies."""
    validators = []
    vid = 0
    
    for _ in range(n_honest):
        stake = max(16, base_stake + np.random.normal(0, stake_variance))
        validators.append(Validator(vid, stake, 'honest'))
        vid += 1
    
    for _ in range(n_occasional):
        stake = max(16, base_stake + np.random.normal(0, stake_variance))
        validators.append(Validator(vid, stake, 'occasionally_malicious'))
        vid += 1
    
    for _ in range(n_aggressive):
        stake = max(16, base_stake + np.random.normal(0, stake_variance))
        validators.append(Validator(vid, stake, 'aggressively_malicious'))
        vid += 1
    
    return validators


np.random.seed(42)
validators = create_validators()
print(f"Created {len(validators)} validators:")
strategy_counts = defaultdict(int)
for v in validators:
    strategy_counts[v.strategy] += 1
for s, c in strategy_counts.items():
    print(f"  {s}: {c} validators")
print(f"Total stake: {sum(v.stake for v in validators):,.1f}")

In [ ]:
# ============================================================
# 5.2 PoS Simulation Engine
# ============================================================

def run_pos_simulation(validators, n_rounds=500, 
                        block_reward=0.1,
                        detection_prob=0.6,
                        slash_fraction=0.10,
                        min_stake=8.0):
    """
    Run a Proof-of-Stake simulation.
    
    Each round:
    1. Select a validator proportional to stake
    2. Validator decides action (honest or malicious)
    3. If honest: earn block_reward
    4. If malicious and detected: slash slash_fraction of stake
    5. Eject validators below min_stake
    """
    # Tracking metrics
    history = {
        'total_stake': [],
        'active_validators': [],
        'honest_count': [],
        'occasional_count': [],
        'aggressive_count': [],
        'slash_events': [],
        'honest_avg_stake': [],
        'malicious_avg_stake': []
    }
    
    cumulative_slashes = 0
    
    for round_num in range(n_rounds):
        active = [v for v in validators if v.active]
        if len(active) == 0:
            break
        
        # Proportional selection
        stakes = np.array([v.stake for v in active])
        probs = stakes / stakes.sum()
        selected_idx = np.random.choice(len(active), p=probs)
        selected = active[selected_idx]
        
        # Action
        action = selected.decide_action()
        slash_this_round = 0
        
        if action == 'honest':
            selected.stake += block_reward
            selected.total_rewards += block_reward
            selected.blocks_produced += 1
        else:
            # Malicious action
            if np.random.random() < detection_prob:
                # Caught! Slash
                penalty = selected.stake * slash_fraction
                selected.stake -= penalty
                selected.total_slashed += penalty
                selected.times_slashed += 1
                slash_this_round = 1
                cumulative_slashes += 1
            else:
                # Not caught, still gets reward
                bonus = block_reward * 1.5  # Malicious behavior yields more if undetected
                selected.stake += bonus
                selected.total_rewards += bonus
                selected.blocks_produced += 1
        
        # Eject validators below min stake
        for v in active:
            if v.stake < min_stake:
                v.active = False
        
        # Record metrics
        active_now = [v for v in validators if v.active]
        history['total_stake'].append(sum(v.stake for v in active_now))
        history['active_validators'].append(len(active_now))
        history['honest_count'].append(
            sum(1 for v in active_now if v.strategy == 'honest'))
        history['occasional_count'].append(
            sum(1 for v in active_now if v.strategy == 'occasionally_malicious'))
        history['aggressive_count'].append(
            sum(1 for v in active_now if v.strategy == 'aggressively_malicious'))
        history['slash_events'].append(cumulative_slashes)
        
        honest_stakes = [v.stake for v in active_now if v.strategy == 'honest']
        malicious_stakes = [v.stake for v in active_now if v.strategy != 'honest']
        history['honest_avg_stake'].append(
            np.mean(honest_stakes) if honest_stakes else 0)
        history['malicious_avg_stake'].append(
            np.mean(malicious_stakes) if malicious_stakes else 0)
    
    return history


# Run simulation
np.random.seed(42)
validators = create_validators()  # Fresh validators
history = run_pos_simulation(validators, n_rounds=500)

print(f"Simulation complete: {len(history['total_stake'])} rounds")
print(f"Final active validators: {history['active_validators'][-1]}")
print(f"Total slashing events: {history['slash_events'][-1]}")

In [ ]:
# ============================================================
# 5.3 Plot Simulation Results
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
rounds = range(len(history['total_stake']))

# Plot 1: Total Stake
ax = axes[0, 0]
ax.plot(rounds, history['total_stake'], color='#2196F3', linewidth=2)
ax.set_title('Total Network Stake Over Time', fontsize=12)
ax.set_xlabel('Round')
ax.set_ylabel('Total Stake')

# Plot 2: Active Validators by Strategy
ax = axes[0, 1]
ax.plot(rounds, history['honest_count'], label='Honest', 
        color='#4CAF50', linewidth=2)
ax.plot(rounds, history['occasional_count'], label='Occasionally Malicious', 
        color='#FF9800', linewidth=2)
ax.plot(rounds, history['aggressive_count'], label='Aggressively Malicious', 
        color='#F44336', linewidth=2)
ax.set_title('Active Validators by Strategy', fontsize=12)
ax.set_xlabel('Round')
ax.set_ylabel('Count')
ax.legend(fontsize=9)

# Plot 3: Average Stake by Type
ax = axes[1, 0]
ax.plot(rounds, history['honest_avg_stake'], label='Honest Avg Stake', 
        color='#4CAF50', linewidth=2)
ax.plot(rounds, history['malicious_avg_stake'], label='Malicious Avg Stake', 
        color='#F44336', linewidth=2)
ax.set_title('Average Stake: Honest vs Malicious', fontsize=12)
ax.set_xlabel('Round')
ax.set_ylabel('Average Stake')
ax.legend(fontsize=10)

# Plot 4: Cumulative Slashing Events
ax = axes[1, 1]
ax.plot(rounds, history['slash_events'], color='#9C27B0', linewidth=2)
ax.set_title('Cumulative Slashing Events', fontsize=12)
ax.set_xlabel('Round')
ax.set_ylabel('Total Slashes')

plt.suptitle('Agent-Based PoS Simulation (50 Validators, 500 Rounds)', 
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5.4 Post-Simulation Analysis
# ============================================================

# Final validator statistics
results = []
for v in validators:
    results.append({
        'ID': v.vid,
        'Strategy': v.strategy,
        'Active': v.active,
        'Final Stake': v.stake,
        'Total Rewards': v.total_rewards,
        'Total Slashed': v.total_slashed,
        'Blocks Produced': v.blocks_produced,
        'Times Slashed': v.times_slashed
    })

df_results = pd.DataFrame(results)

# Summary by strategy
summary = df_results.groupby('Strategy').agg({
    'Active': 'sum',
    'Final Stake': 'mean',
    'Total Rewards': 'mean',
    'Total Slashed': 'mean',
    'Blocks Produced': 'mean',
    'Times Slashed': 'mean'
}).round(2)

summary.columns = ['Still Active', 'Avg Final Stake', 'Avg Rewards', 
                    'Avg Slashed', 'Avg Blocks', 'Avg Times Slashed']

print("Post-Simulation Summary by Strategy")
print("=" * 80)
print(summary.to_string())
print()

# Survival rates
print("Survival Rates:")
for strategy in df_results['Strategy'].unique():
    group = df_results[df_results['Strategy'] == strategy]
    survival = group['Active'].mean() * 100
    print(f"  {strategy}: {survival:.0f}% survival rate")

### Agent-Based Simulation Insights

The simulation demonstrates several key properties of well-designed PoS systems:

1. **Honest validators prosper**: Over time, honest validators accumulate more stake through consistent rewards without slashing risk.

2. **Malicious validators are gradually ejected**: The combination of slashing penalties and minimum stake requirements creates natural selection pressure against dishonest behavior.

3. **The network becomes more secure over time**: As malicious validators are removed, the proportion of honest stake increases, making attacks progressively harder.

4. **Slashing is the key mechanism**: Without slashing, the cost of attempting attacks approaches zero (the "nothing at stake" problem). Slashing creates real economic consequences.

---

# Exercises

Complete the following exercises to deepen your understanding of cryptoeconomic modeling.

## Exercise 1: Add a "Grudger" Strategy to the PD Tournament

The **Grudger** (also called "Grim Trigger") strategy cooperates until the opponent defects once, then defects forever. Implement this strategy and add it to the tournament.

How does Grudger compare to Tit-for-Tat? In what situations does it perform better or worse?

In [ ]:
# ============================================================
# Exercise 1: Grudger Strategy
# ============================================================

def grudger(history_self, history_opponent):
    """Cooperate until opponent defects, then defect forever."""
    # TODO: Implement the grudger strategy
    # Hint: Check if DEFECT ever appears in history_opponent
    pass


# Add to strategies and re-run tournament
# strategies_extended = {**strategies, 'Grudger': grudger}
# np.random.seed(42)
# total_scores_ext, match_results_ext = run_tournament(strategies_extended, rounds=200)

# Print results
# for name, score in sorted(total_scores_ext.items(), key=lambda x: -x[1]):
#     print(f"  {name:20s}: {score:6.0f} points")

## Exercise 2: Model Nothing-at-Stake and Show Slashing Solves It

The **nothing-at-stake** problem occurs in PoS when validators can vote on multiple competing forks at no cost. Without slashing, the rational strategy is to vote on *every* fork (hedging bets), which undermines consensus.

Model a scenario with 2 competing forks and show:
1. Without slashing, validators always vote on both forks
2. With slashing for double-voting, validators converge on one fork

In [ ]:
# ============================================================
# Exercise 2: Nothing-at-Stake Problem
# ============================================================

def nothing_at_stake_simulation(n_validators=20, n_rounds=100, 
                                 slashing_enabled=False,
                                 slash_penalty=0.3,
                                 detection_prob=0.8):
    """
    Simulate nothing-at-stake with two competing forks.
    
    Each round, validators choose:
    - Vote for Fork A only
    - Vote for Fork B only  
    - Vote for BOTH forks (nothing-at-stake)
    
    TODO: Implement the simulation
    - Without slashing: voting both is always optimal (double rewards, no risk)
    - With slashing: double-voting is detected and penalized
    
    Returns: history of fork_a_votes, fork_b_votes, double_votes per round
    """
    # TODO: Implement
    pass


# Run with and without slashing, compare results
# history_no_slash = nothing_at_stake_simulation(slashing_enabled=False)
# history_with_slash = nothing_at_stake_simulation(slashing_enabled=True)

## Exercise 3: Validator ROI Calculator with Variable Parameters

Build a comprehensive ROI calculator that takes:
- Initial stake amount and token price
- Staking yield (APR)
- Hardware and bandwidth costs
- Token price appreciation/depreciation scenarios
- Compounding frequency

Output a 3-year projection table and break-even analysis.

In [ ]:
# ============================================================
# Exercise 3: Validator ROI Calculator
# ============================================================

def validator_roi_calculator(initial_stake, token_price_usd, 
                              staking_apr_pct, monthly_costs_usd,
                              price_change_annual_pct=0.0,
                              compound_frequency='monthly',
                              projection_months=36):
    """
    Calculate validator ROI over time.
    
    TODO: Implement this function
    
    Returns: DataFrame with monthly projections including:
    - Month, Stake (tokens), Token Price, Portfolio Value (USD),
    - Cumulative Costs (USD), Net Profit (USD), ROI (%)
    """
    # TODO: Implement
    pass


# Example usage:
# df_roi = validator_roi_calculator(
#     initial_stake=32, token_price_usd=3200,
#     staking_apr_pct=4.0, monthly_costs_usd=100,
#     price_change_annual_pct=20.0
# )
# print(df_roi.to_string(index=False))

## Exercise 4: Simulate Liquidity Network Effects in a DEX

Decentralized exchanges (DEXs) exhibit strong network effects through liquidity: more liquidity means tighter spreads, which attracts more traders, which attracts more liquidity providers.

Simulate a simplified DEX with:
- Liquidity providers (LPs) who earn fees
- Traders who benefit from lower slippage with deeper pools
- A feedback loop: more LPs -> less slippage -> more traders -> more fees -> more LPs

In [ ]:
# ============================================================
# Exercise 4: DEX Liquidity Network Effects
# ============================================================

def simulate_dex_liquidity(n_periods=200, 
                            initial_liquidity=100_000,
                            initial_daily_volume=10_000,
                            fee_rate=0.003,  # 0.3%
                            lp_sensitivity=0.5,
                            trader_sensitivity=0.3):
    """
    Simulate DEX liquidity and volume dynamics.
    
    TODO: Implement the feedback loop:
    1. Slippage decreases as liquidity increases: slippage ~ volume / liquidity
    2. Volume increases as slippage decreases: volume_growth ~ -slippage * trader_sensitivity
    3. LP returns increase with volume: lp_return = volume * fee_rate / liquidity
    4. Liquidity grows when LP returns are attractive: liq_growth ~ lp_return * lp_sensitivity
    
    Returns: liquidity, volume, slippage, lp_apy over time
    """
    # TODO: Implement
    pass


# Run simulation and plot results
# results = simulate_dex_liquidity()
# Plot liquidity, volume, slippage, and LP APY over time

---

# Summary

In this notebook, we built computational models of the core cryptoeconomic mechanisms that make blockchains work:

## Key Takeaways

1. **Game Theory** (Part 1): Blockchain consensus is fundamentally a repeated game. Tit-for-Tat-like strategies (cooperate by default, punish defection) emerge naturally in protocol design.

2. **Nash Equilibrium** (Part 2): Honest mining is not always a Nash Equilibrium. Selfish mining becomes rational at ~25% hash power, motivating the need for additional protocol safeguards.

3. **Staking Economics** (Part 3): Validator returns depend on participation rates, inflation, hardware costs, and slashing parameters. Real yields can differ significantly from nominal yields. Break-even analysis reveals the economic barriers to solo validation.

4. **Network Effects** (Part 4): Metcalfe's Law creates powerful winner-take-most dynamics. S-curve adoption and two-sided market dynamics explain blockchain growth patterns.

5. **Agent-Based Simulation** (Part 5): PoS systems with slashing create evolutionary pressure against malicious validators. Over hundreds of rounds, honest validators accumulate stake while malicious ones are gradually ejected.

## Next Steps

- Review `sections/04-blockchain-economics.md` for the theoretical foundations underlying these models
- Explore mechanism design: how do you *design* payoff matrices that incentivize honest behavior?
- Study MEV (Maximal Extractable Value) as a real-world application of these game-theoretic concepts
- Investigate cross-chain economic models and interoperability incentives

---

*Notebook 10 of the MIT Sloan Blockchain Education Series*